# Pydantic Tutorial — Validating LLM Output (using Qwen2.5 locally)

Hands-on guide to using [Pydantic](https://docs.pydantic.dev/) as the contract between your code and a **local** Qwen2.5-7B model served by Ollama.

**Setup**: `uv sync` — see the [README](../../../README.md#setup)
**LLM**: Qwen2.5-7B served locally with Ollama — no account, no token, fully offline.

> No data leaves your machine. The `openai` package is only an HTTP client for your local Ollama server at `http://localhost:11434`.

Pull the model before running:
```bash
ollama pull qwen2.5:7b
```

**Covered topics**:
1. Pydantic Basics
2. Validation Errors
3. Schema-Guided Generation
4. The `.parse()` Helper
5. What the Grammar Does and Doesn't Enforce
6. Retry with Validation Feedback
7. Nested Models & Business Rules

---

### The idea in one paragraph

"Structured output" is really **two independent guarantees**, and it pays to keep them apart:

| | Enforced by | Guarantees |
|---|---|---|
| **Shape** | the decoder (Ollama, from your JSON Schema) | valid JSON, right types, required keys present, enum members, array bounds |
| **Meaning** | Pydantic, after the fact | numeric ranges, cross-field rules, anything domain-specific |

A grammar can only make output *well-formed*. It cannot make it *true*. Section 5 shows precisely where the two part company — that gap is the reason you still validate output a grammar already "guaranteed".

In [ ]:
# Dependencies come from the repo's uv environment. Run once in a terminal:
#     uv sync
#
# Then `uv run jupyter lab`, or point your IDE's kernel at .venv/bin/python.
# Syncing a new group while this kernel is running? Restart the kernel after.

## Setup

One OpenAI-compatible client pointed at Ollama, plus two helpers reused throughout:

- `structured_call()` sends a Pydantic model's JSON Schema as the response format and returns the **raw, unvalidated** string — so we can inspect output that is well-formed but wrong.
- `describe_errors()` flattens a `ValidationError` into one line per field, which is exactly what we feed back to the model in Section 6.

In [2]:
import json

from openai import OpenAI
from pydantic import (
    BaseModel,
    EmailStr,
    Field,
    ValidationError,
    field_validator,
    model_validator,
)
from typing import Literal, Optional

MODEL = "qwen2.5:7b"

client = OpenAI(
    base_url="http://localhost:11434/v1",  # local Ollama server
    api_key="ollama",                      # required by the client, ignored by Ollama
)


def structured_call(schema_model, messages, temperature=0.0):
    """Ask the LLM for JSON constrained by a Pydantic model's JSON Schema."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": schema_model.__name__,
                "schema": schema_model.model_json_schema(),
                "strict": True,
            },
        },
    )
    return response.choices[0].message.content


def describe_errors(exc: ValidationError) -> str:
    """Flatten a ValidationError into one line per offending field."""
    parts = []
    for err in exc.errors():
        location = ".".join(str(p) for p in err["loc"]) or "(whole model)"
        parts.append(f"{location}: {err['msg']}")
    return "; ".join(parts)

The model we will ask the LLM to fill in for most of the notebook. Every constraint here is a promise we intend to hold the model to:

In [3]:
class MovieRecommendation(BaseModel):
    """The contract we want every movie recommendation to satisfy."""

    title: str = Field(min_length=1, max_length=100)
    genre: Literal[
        "action", "comedy", "drama", "sci-fi", "thriller", "horror", "romance"
    ]
    year: int = Field(ge=1900, le=2026)
    rating: float = Field(ge=0.0, le=10.0, description="Rating from 0 to 10")
    synopsis: str = Field(min_length=10, max_length=500)
    director: Optional[str] = None
    recommended_for: Optional[Literal["family", "adults", "teens"]] = None

## 1. Pydantic Basics

A Pydantic model is a class inheriting from `BaseModel` where each annotated attribute becomes a validated field. `Field()` attaches constraints (`ge`, `le`, `min_length`, …) and documentation.

Validation runs on construction: if the data does not fit, you get an exception rather than a half-built object.

In [4]:
class UserInput(BaseModel):
    name: str
    email: EmailStr
    query: str
    order_id: Optional[int] = Field(None, ge=10000, le=99999)


user = UserInput(
    name="Joe User",
    email="joe.user@example.com",
    query="I forgot my password.",
)
print("Instance: ", user)
print("As JSON:  ", user.model_dump_json())
print("order_id defaulted to:", user.order_id)

Instance:  name='Joe User' email='joe.user@example.com' query='I forgot my password.' order_id=None
As JSON:   {"name":"Joe User","email":"joe.user@example.com","query":"I forgot my password.","order_id":null}
order_id defaulted to: None


**Type coercion.** Pydantic converts values that unambiguously represent the target type. An LLM that returns `"12345"` where you asked for an integer does not break your pipeline:

In [5]:
coerced = UserInput(
    name="Joe User",
    email="joe.user@example.com",
    query="I need help.",
    order_id="12345",  # a string, not an int
)
print(f"Coerced '12345' -> {coerced.order_id!r} ({type(coerced.order_id).__name__})")

Coerced '12345' -> 12345 (int)


**Extra keys are dropped, not rejected.** This is the default (`extra="ignore"`) and it matters for LLM output: a chatty model can bolt on fields it invented and validation still passes, silently discarding them.

If you would rather catch that, set `model_config = ConfigDict(extra="forbid")`.

In [6]:
extra = UserInput(
    name="Joe User",
    email="joe.user@example.com",
    query="I need help.",
    confidence=0.97,           # not on the model
    note="thinking out loud",  # not on the model
)
print("Extra keys dropped:", extra.model_dump_json())

Extra keys dropped: {"name":"Joe User","email":"joe.user@example.com","query":"I need help.","order_id":null}


## 2. Validation Errors

`ValidationError.errors()` returns one dictionary per problem, with:

- `loc` — the field path as a tuple (nested fields give multi-element paths)
- `msg` — the human-readable reason
- `type` — a stable machine-readable identifier

Note that Pydantic reports **all** failures at once, not just the first. That whole list is the useful thing to hand back to a model on a retry.

In [7]:
bad_input = {"name": "Joe User", "email": "not-an-email"}  # `query` missing too

try:
    UserInput(**bad_input)
except ValidationError as exc:
    print(f"{len(exc.errors())} problem(s) found:")
    for err in exc.errors():
        field = ".".join(str(p) for p in err["loc"])
        print(f"  - {field}: {err['msg']}  (type={err['type']})")
    print("\nFlattened for a retry prompt:")
    print(" ", describe_errors(exc))

2 problem(s) found:
  - email: value is not a valid email address: An email address must have an @-sign.  (type=value_error)
  - query: Field required  (type=missing)

Flattened for a retry prompt:
  email: value is not a valid email address: An email address must have an @-sign.; query: Field required


## 3. Schema-Guided Generation

`model_json_schema()` turns your model into a JSON Schema. Ollama accepts that schema as `response_format` and constrains decoding to it, so the reply is guaranteed to parse as JSON of the right shape.

Notice what survives the trip into the schema (`enum`, `maxItems`, `minLength`, `maximum`) — Section 5 checks which of those the decoder actually honours.

In [8]:
schema = MovieRecommendation.model_json_schema()
print(json.dumps(schema, indent=2)[:400], "...")

{
  "description": "The contract we want every movie recommendation to satisfy.",
  "properties": {
    "title": {
      "maxLength": 100,
      "minLength": 1,
      "title": "Title",
      "type": "string"
    },
    "genre": {
      "enum": [
        "action",
        "comedy",
        "drama",
        "sci-fi",
        "thriller",
        "horror",
        "romance"
      ],
      "title": "Ge ...


In [ ]:
raw = structured_call(
    MovieRecommendation,
    [
        {"role": "system", "content": "You are a movie recommendation expert."},
        {"role": "user", "content": "Recommend a sci-fi movie for a relaxing weekend."},
    ],
)
print("Raw model output:")
print(raw)

movie = MovieRecommendation.model_validate_json(raw)
print(f"\nValidated -> {movie.title} ({movie.year}), {movie.rating}/10")

## 4. The `.parse()` Helper

The OpenAI SDK bundles both steps: pass the model class itself as `response_format` and `.parse()` sends the schema *and* validates the reply, handing back a typed object.

> On older SDKs this lived at `client.beta.chat.completions.parse()`. The `beta` path still works, but `client.chat.completions.parse()` is the current one.

In [ ]:
completion = client.chat.completions.parse(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a movie recommendation expert."},
        {"role": "user", "content": "Suggest a comedy suitable for family viewing."},
    ],
    response_format=MovieRecommendation,
    temperature=0,
)

movie = completion.choices[0].message.parsed
print("Type:", type(movie).__name__)
print(f"{movie.title} ({movie.year}) — {movie.genre}, {movie.rating}/10")
print("Recommended for:", movie.recommended_for)

The convenience has a sharp edge. Validation happens **inside** the call, so a constraint violation surfaces as a `ValidationError` raised at the call site — there is no half-valid response object to inspect. Always wrap it.

Below we invite the failure by asking for a 0–100 rating while the model caps `rating` at 10:

In [ ]:
try:
    completion = client.chat.completions.parse(
        model=MODEL,
        messages=[
            {"role": "user", "content": "Rate the movie Inception on a scale from 0 to 100."}
        ],
        response_format=MovieRecommendation,
        temperature=0,
    )
    print("Parsed:", completion.choices[0].message.parsed)
except ValidationError as exc:
    print("ValidationError raised by .parse():", describe_errors(exc))
    print("The rating came back on a 0-100 scale; the model is capped at 10.")

## 5. What the Grammar Does and Doesn't Enforce

This is the section worth internalising. The decoder honours some of your schema and quietly ignores the rest — and where it *does* enforce, "enforced" is not the same as "correct".

Four probes, each deliberately asking for something the schema forbids.

### (a) Enum members are enforced — into a confidently wrong answer

Asked for a genre outside the `Literal`, the decoder cannot emit `"documentary"`, so it picks an allowed value instead. The result is schema-perfect and factually wrong. Enum constraints convert "the model does not know" into "the model states something false", which no amount of downstream validation can detect.

In [ ]:
class Genre(BaseModel):
    genre: Literal["action", "comedy", "drama"]


raw = structured_call(
    Genre,
    [{"role": "user", "content": "The genre of 'Planet Earth' is documentary. Return it."}],
)
print("asked for 'documentary' ->", raw.strip())

### (b) Array bounds are enforced

`maxItems` comes straight from `Field(max_length=3)` on a list, and the decoder respects it — 8 tags requested, 3 returned.

In [ ]:
class Tags(BaseModel):
    tags: list[str] = Field(min_length=1, max_length=3)


raw = structured_call(
    Tags, [{"role": "user", "content": "Give exactly 8 tags describing the ocean."}]
)
print("asked for 8 tags ->", raw.strip())

### (c) Numeric ranges are **not** enforced

`minimum` / `maximum` reach the schema and are then ignored during decoding. Ask for a 0–100 rating against a `le=10.0` field and you will usually get something like `93`.

This is the single most important gap: it is the reason a Pydantic pass after generation is not redundant. (Occasionally the model complies on its own — rerun the cell a couple of times.)

In [ ]:
class Score(BaseModel):
    title: str
    score: float = Field(ge=0.0, le=10.0)


raw = structured_call(
    Score,
    [{"role": "user", "content": "Rate the movie Inception from 0 to 100."}],
    temperature=0.9,
)
print("asked for 0-100 ->", raw.strip())

try:
    Score.model_validate_json(raw)
    print("pydantic: accepted (the model happened to comply this time)")
except ValidationError as exc:
    print("pydantic REJECTS:", describe_errors(exc))

### (d) String length is enforced — destructively

`minLength` **is** applied during decoding, and that is worse than ignoring it. The decoder is not allowed to close the string before 200 characters, so a model that wanted to answer in three words pads its way to the quota with whatever tokens come next — often visible garbage.

The punchline: that output satisfies the schema *and* passes Pydantic. Length was the wrong tool for "write a thorough summary".

In [ ]:
class Summary(BaseModel):
    summary: str = Field(min_length=200, max_length=500)


raw = structured_call(
    Summary, [{"role": "user", "content": "Summarize the ocean in exactly three words."}]
)
text = json.loads(raw)["summary"]

print(f"asked for 3 words -> {len(text)} chars")
print(text[:160], "...")
print("\npadded to satisfy the grammar — and Pydantic accepts it:",
      Summary.model_validate_json(raw) is not None)

### Summary of the four probes

| Constraint | In the schema | Enforced while decoding? | What actually happens |
|---|---|---|---|
| `Literal[...]` → `enum` | ✅ | ✅ | Coerced into the allowed set — can be confidently wrong |
| `max_length` on a list → `maxItems` | ✅ | ✅ | Respected cleanly |
| `ge` / `le` → `minimum` / `maximum` | ✅ | ❌ | Freely violated — **Pydantic catches it** |
| `min_length` on a str → `minLength` | ✅ | ✅ | Padded to quota with junk that then validates |

**Rule of thumb:** let the grammar fix the *shape*, let Pydantic judge the *meaning*, and keep length and range wishes out of the decoder — express them as constraints you validate (Section 6) or as prose in the prompt.

## 6. Retry with Validation Feedback

Since ranges survive decoding unchecked, the practical loop is: generate → validate → on failure, hand the model its own errors and ask for a correction.

One trap worth knowing: at `temperature=0` a rejected answer is regenerated **verbatim**, and the loop spins until it hits the attempt cap. Nudging the temperature on retries is what actually breaks the tie — you can watch attempts 1 and 2 come back identical below.

In [ ]:
messages = [{"role": "user", "content": "Rate the movie Inception from 0 to 100."}]

for attempt in range(1, 4):
    raw = structured_call(Score, messages, temperature=0.0 if attempt == 1 else 0.4)
    print(f"Attempt {attempt}: {raw.strip()}")

    try:
        score = Score.model_validate_json(raw)
        print(f"  accepted -> {score.title}: {score.score}/10")
        break
    except ValidationError as exc:
        reasons = describe_errors(exc)
        print(f"  rejected -> {reasons}")
        messages += [
            {"role": "assistant", "content": raw},
            {
                "role": "user",
                "content": (
                    f"That response failed validation: {reasons}. "
                    "Fix only those fields and return corrected JSON."
                ),
            },
        ]
else:
    print("  gave up after 3 attempts")

## 7. Nested Models & Business Rules

Models compose: a field typed as another `BaseModel` becomes a `$ref` in the schema, and Ollama handles those fine. This is what makes real extraction work — one call turning an unstructured email into typed objects you can compute on.

Two validator hooks cover what a schema cannot express:

- `@field_validator` — runs per field after coercion. Good for normalising (trimming, lower-casing).
- `@model_validator(mode="after")` — runs once with the whole object, so it can compare fields against each other.

In [ ]:
class LineItem(BaseModel):
    product: str
    quantity: int = Field(ge=1)
    unit_price: float = Field(ge=0)


class SupportTicket(BaseModel):
    """Nested extraction target, with rules a schema alone cannot express."""

    customer_email: EmailStr
    category: Literal["refund_request", "information_request", "complaint", "other"]
    priority: Literal["low", "medium", "high"]
    is_complaint: bool
    items: list[LineItem]
    tags: list[str] = Field(max_length=4)
    order_id: Optional[int] = Field(None, ge=10000, le=99999)

    @field_validator("tags")
    @classmethod
    def normalise_tags(cls, tags: list[str]) -> list[str]:
        return [tag.strip().lower().replace(" ", "_") for tag in tags]

    @model_validator(mode="after")
    def complaints_are_never_low_priority(self):
        if self.is_complaint and self.priority == "low":
            raise ValueError("a complaint cannot be priority 'low'")
        return self


print("schema uses $defs for the nested model:", "$defs" in SupportTicket.model_json_schema())

In [ ]:
email = (
    "From: joe.user@example.com\n"
    "I ordered 2 mechanical keyboards at 89.99 each and 1 monitor stand at "
    "34.50 on order 12345. The monitor stand arrived cracked and I was "
    "charged twice. I want a refund immediately, this is unacceptable."
)

raw = structured_call(
    SupportTicket,
    [
        {"role": "system", "content": "Extract a structured support ticket from the email."},
        {"role": "user", "content": email},
    ],
)
print(raw)

Once validated it is an ordinary Python object — the line items are `LineItem` instances, so arithmetic just works:

In [ ]:
ticket = SupportTicket.model_validate_json(raw)

print(f"Category: {ticket.category} | priority: {ticket.priority}")
print(f"Order:    {ticket.order_id} | tags: {ticket.tags}")
for item in ticket.items:
    print(f"  {item.quantity} x {item.product} @ {item.unit_price}")

total = sum(item.quantity * item.unit_price for item in ticket.items)
print(f"Order total: {total:.2f}")

And the cross-field rule fires on data that the JSON Schema considers flawless — every field is the right type and an allowed value, yet the combination is nonsense for the business:

In [ ]:
try:
    SupportTicket(
        customer_email="joe.user@example.com",
        category="complaint",
        priority="low",
        is_complaint=True,
        items=[LineItem(product="monitor stand", quantity=1, unit_price=34.5)],
        tags=["Broken Item"],
    )
except ValidationError as exc:
    print("Business rule caught it:", describe_errors(exc))

## Summary

- **`BaseModel` + `Field`** define the contract; validation runs on construction and reports every failure at once.
- **Type coercion** absorbs harmless mismatches (`"12345"` → `12345`); **extra keys are dropped silently** unless you set `extra="forbid"`.
- **`model_json_schema()` → `response_format`** constrains decoding, so the reply always parses.
- **`client.chat.completions.parse()`** does schema + validation in one step and raises `ValidationError` at the call site.
- **The decoder honours shape, not meaning.** Enums and array bounds hold; numeric ranges do not; string lengths hold by padding junk. Pydantic is the gate that catches the difference.
- **Validation errors are a prompt**: flatten `exc.errors()`, feed them back, raise the temperature so the retry can differ.
- **`field_validator` / `model_validator`** carry the rules a schema cannot express.

### Where to go next

- Swap `Literal` for an `Enum` when you need the values elsewhere in your code.
- Add `Annotated[str, StringConstraints(...)]` for reusable constrained types across models.
- Push the validated objects into the RAG pipeline from the [LangChain tutorial](../../langchain/notebooks/langchain-tutorial.ipynb), or serve the same models behind vLLM's guided decoding ([vLLM tutorial](../../vllm/notebooks/vllm-tutorial.ipynb)).
- Use the models as graders and score them with the [DeepEval](../../../evaluation/deepeval/notebooks/deepeval-evaluation-tutorial.ipynb) or [RAGAS](../../../evaluation/ragas/notebooks/ragas-evaluation-tutorial.ipynb) tutorials.

**Reference**: [`inference/pydantic/pydantic-reference.md`](../pydantic-reference.md) — API cheatsheet for the constructs used here.